In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import scipy
import numba
import molsim

<div style="max-width: 1000px; margin-left: 0; margin-right: auto; font-size: 20px; line-height: 1.6;">

# Exercise 1: Distribution of particles

Consider an ideal gas of N particles in a constant volume at constant energy. Let us divide the volume in p identical compartments. Every compartment contains ni molecules such that

\begin{equation}
N = \sum^{i=p}_{i=1}n_i
\end{equation}

An interesting quantity is the distribution of molecules over the $p$ compartments. Because the energy is constant, every possible eigenstate of the system will be equally likely. This means that in principle it is possible that one of the compartments is empty.

<div style="max-width: 1000px; margin-left: 0; margin-right: auto; font-size: 20px; line-height: 1.6;">

# Question 1:
Show that the maximum of a multinomial distribution is given when $N_1$=$N_2$=...=$N_p$ = $N/p$  
(D. McQuarrie, prob 1-50)

<div style="max-width: 1000px; margin-left: 0; margin-right: auto; font-size: 20px; line-height: 1.6;">

# Question 2: 
Below you will find code that calculates the distribution of molecules along the $p$ compartments. Note that the code that loops over the particles and puts them in the compartments has to be completed first. Run the program for different numbers of compartments ($p$) and total number of gas molecules ($N$). The plotting function can be run to visualize the distribution of the particles

Implement the algorithm for randomly placing the particles in a compartment below. Use loops over the number of cycles and particles and devise a way to randomly select a compartment. 

</div>

In [ ]:

# start refactor
numberOfParticles = 1000
numberOfCompartments = 10
# end refactor
numberOfCycles = 1000

probabilities = np.ones(numberOfCompartments) / numberOfCompartments

particlesInCompartment = np.zeros((numberOfCycles, numberOfCompartments), dtype=np.int32)

In [ ]:
#**********************************************************************************
# start implementation
   
# end implementation
#**********************************************************************************

In [ ]:
def lastNonZero(series: np.ndarray):
    # Find the index of the last non-zero element in the series
    return len(series) - np.where(series[::-1] > 0)[0][0]


def plotDistribution(
    particlesInCompartment: np.ndarray,
    numberOfParticles: int,
    analyticalParticleDistribution: np.ndarray = None,
    analyticalCompartmentDistribution: np.ndarray = None,
):
    """Plot the distribution of particles in compartments

    Paramters:
        - particlesInCompartment (np.ndarray): Array containing the number of particles in each compartment for each cycle.
        - numberOfParticles (int): Total number of particles.
        - analyticalParticleDistribution (np.ndarray, optional): Analytical distribution of particles. Defaults to None.
        - analyticalCompartmentDistribution (np.ndarray, optional): Analytical distribution of compartments. Defaults to None.

    Returns:
        - None
    """

    numberOfCompartments = particlesInCompartment.shape[1]
    colors = plt.cm.cividis(np.linspace(1, 0, numberOfCompartments))

    # Create a figure with a custom gridspec
    fig = plt.figure(figsize=(12, 8))
    gs = gridspec.GridSpec(2, 2, height_ratios=[2, 1])  # Two rows, two columns, bottom row is taller

    # Create the subplots
    ax0 = fig.add_subplot(gs[0, 0])  # Top-left
    ax1 = fig.add_subplot(gs[0, 1])  # Top-right
    ax2 = fig.add_subplot(gs[1, :])  # Bottom, spanning both columns

    # Calculate the distribution of particles in each compartment
    totalDistribution = np.zeros(numberOfParticles)
    for c in range(numberOfCompartments):
        distribution, _ = np.histogram(
            particlesInCompartment[:, c], bins=numberOfParticles, range=(-0.5, numberOfParticles + 0.5), density=True
        )
        ax0.plot(distribution, label=f"$E_{{{c}}}$", c=colors[c], lw=1)
        totalDistribution += distribution

    # Plot the analytical distribution if provided
    if analyticalParticleDistribution is not None:
        ax0.plot(analyticalParticleDistribution, label="Analytical", c="black")

    ax0.set_xlabel("Number of particles", fontsize=20)
    ax0.set_ylabel("Density of probability", fontsize=20)
    ax0.set_xlim(0, lastNonZero(totalDistribution))
    ax0.legend(fontsize=12)

    # Plot the distribution of particles in each compartment
    ax1.boxplot(
        particlesInCompartment,
        patch_artist=True,
        boxprops=dict(facecolor="skyblue"),
        tick_labels=np.arange(numberOfCompartments),
    )
    ax1.set_xlabel("Compartment", fontsize=20)
    ax1.set_ylabel("Number of particles", fontsize=20)
    if analyticalCompartmentDistribution is not None:
        ax1.plot(np.arange(numberOfCompartments) + 1, analyticalCompartmentDistribution, label="Analytical", c="black")
    ax1.grid(False)
    ax1.legend()

    # Plot the distribution of particles in the last cycle
    ax2.axis("off")
    ax2.text(0.3, 0.0, "Distribution in last cycle", fontsize=28)
    for i, val in enumerate(particlesInCompartment[-1]):
        ax2.add_patch(
            plt.Rectangle(
                (i / numberOfCompartments, 0.25),
                1 / (numberOfCompartments + 1),
                0.5,
                edgecolor="black",
                facecolor="skyblue",
            )
        )
        ax2.text(
            ((i + 0.45) / numberOfCompartments), 0.5, str(val), fontsize=24, ha="center", va="center", color="black"
        )

    fig.tight_layout()


def analyticalParticleDistribution(numberOfParticles: int, numberOfCompartments: int):
    """Calculate the analytical distribution of particles in compartments"""

    # get index array
    idx = np.arange(numberOfParticles)

    # get binomial coefficients npr(N, j) in array
    binomCoeffs = scipy.special.binom(numberOfParticles, idx)

    # precompute p
    p = 1 / numberOfCompartments

    # return binomial pmf
    return binomCoeffs * p**idx * (1 - p) ** (numberOfParticles - idx)


plotDistribution(
    particlesInCompartment,
    numberOfParticles,
    analyticalParticleDistribution(numberOfParticles, numberOfCompartments),
    np.ones(numberOfCompartments) * (numberOfParticles / numberOfCompartments),
)

<div style="max-width: 1000px; margin-left: 0; margin-right: auto; font-size: 20px; line-height: 1.6;">

# Question 3:

Why does it (almost) never happen that one of the compartments is empty when $N/P ≫ 1$?

# Exercise 2:

We consider a polymer consisting of three beads connected by harmonic springs.

<img src="polymer.png" width="450">

For the polymer ilustrated in the figure below, estimate the partition function $\Omega (N,V,E)$. Asume this molecule is in a box of volume V and that the equilibrium bond length is equal to zero. Remeber the potential energy for a polymer contaning N beads can be written as: 

$
U(\mathbf{r_1},...,\mathbf{r_N})=\frac{1}{2}m\omega \sum_{k=1}^N (r_k-r_{k+1})^2
$

and use that 

$
\Omega(N,V,E) =
\frac{E_0}{N!\,\Gamma\!\left(\frac{3N}{2}\right)}
\left[
\left(\frac{2\pi m}{h^2}\right)^{3/2}
\right]^N
\times
\int_{D(V)} \mathbf{d^N r} \;
\left[E - U(r_1,\ldots,r_N)\right]^{\frac{3N}{2}-1}
\theta\!\left(E - U(r_1,\ldots,r_N)\right)
$

where $\theta (x)$ is the Heaviside step function. What is the total temperature $T$ of this polymer? Show that your result matches the one that can be obtained from the equipartition theorem ($T=\frac{2E}{15k_B}$) (Hint: use reduce coordinates to express atomic coordinates. Also, see Stat. Mech. by Tuckerman, Chap 3, prob. 3.2) 